In [13]:
import joblib 
import dice_ml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# pick which model to explain — comment out the others
#TAG = "augmented"
#TAG = "freegen_qwen"
#TAG = "freegen_llama"
#TAG = "rewrite_qwen"
TAG = "rewrite_llama"

b = joblib.load(f"xgb_bundle_{TAG}.joblib")
clf, tfidf         = b["clf"], b["tfidf"]
feat_names, hand_names = b["feat_names"], b["hand_names"]
Xte, te_x, te_y    = b["Xte"], b["te_x"], np.array(b["te_y"])
hte, htr, tr_y     = b["hte"], b["htr"], np.array(b["tr_y"])
prob, pred         = np.array(b["prob"]), np.array(b["pred"])
print(f"Loaded {TAG}: {Xte.shape[0]} test notes, {len(feat_names)} features")

Loaded rewrite_llama: 400 test notes, 513 features


# Q4: What would it take to flip a synthetic note to real?

In [14]:
from xgboost import XGBClassifier

clf_hand = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=2.0,
    eval_metric="logloss", random_state=42).fit(htr.values, tr_y)

# take one confidently-caught synthetic note
syn_rows = np.where((te_y == 1) & (pred == 1))[0]
i = int(syn_rows[np.argmax(prob[syn_rows])])
x0 = hte.values[i].copy()
print(f"{TAG}: original P(synthetic) = {clf_hand.predict_proba([x0])[0,1]:.3f}\n")

# set each feature to the real-class mean, one at a time; see which most reduces synthetic-ness
real_mean = htr.values[tr_y == 0].mean(axis=0)
rows = []
for j in range(len(hand_names)):
    x = x0.copy(); x[j] = real_mean[j]
    rows.append((hand_names[j], x0[j], real_mean[j], clf_hand.predict_proba([x])[0,1]))

print("Set feature → real-class mean (one at a time):")
for name, orig, tgt, p in sorted(rows, key=lambda t: t[3]):
    flag = "  → flips to REAL" if p < 0.5 else ""
    print(f"  {name:24s} {orig:9.2f} → {tgt:9.2f}   P(syn)={p:.3f}{flag}")

rewrite_llama: original P(synthetic) = 0.999

Set feature → real-class mean (one at a time):
  flesch                        8.37 →     42.43   P(syn)=0.917
  uppercase_word_ratio          0.03 →      0.15   P(syn)=0.922
  word_count                  583.00 →   1593.79   P(syn)=0.936
  sentence_count               46.00 →    148.58   P(syn)=0.992
  vocab_richness                0.72 →      0.49   P(syn)=0.995
  connector_density             0.51 →      0.09   P(syn)=0.998
  comma_density                 8.78 →      6.82   P(syn)=0.999
  paren_density                 0.92 →      2.13   P(syn)=0.999
  avg_sentence_length          12.89 →     11.42   P(syn)=0.999
  sentence_length_std          12.40 →     12.39   P(syn)=0.999
  sentence_length_cv            0.96 →      1.08   P(syn)=0.999
  colon_density                 8.32 →      8.39   P(syn)=0.999
  semicolon_density             0.00 →      0.19   P(syn)=0.999


In [15]:
train_df = htr.copy()
train_df["label"] = tr_y

d = dice_ml.Data(
    dataframe=train_df,
    continuous_features=list(htr.columns),
    outcome_name="label",
)
m = dice_ml.Model(model=clf_hand, backend="sklearn")   # clf_hand from your PDP step
exp = dice_ml.Dice(d, m, method="random")   # 'random' is simplest; 'genetic' more thorough

# pick a confidently-synthetic note
syn_rows = np.where((te_y == 1) & (pred == 1))[0]
i = int(syn_rows[np.argmax(prob[syn_rows])])
query = hte.iloc[[i]]

# only let it change style features you consider mutable; freeze the rest
mutable = ["vocab_richness","flesch","uppercase_word_ratio",
           "connector_density","sentence_length_cv","avg_sentence_length"]

cf = exp.generate_counterfactuals(
    query, total_CFs=3, desired_class=0,          # flip to "real"
    features_to_vary=mutable,
)
cf.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00, 18.85it/s]

Query instance (original outcome : 1)


,word_count,sentence_count,avg_sentence_length,sentence_length_std,sentence_length_cv,vocab_richness,uppercase_word_ratio,comma_density,colon_density,semicolon_density,paren_density,flesch,connector_density,label
0,583,46,12.891304,12.403065,0.962126,0.723842,0.03259,8.784096,8.321775,0.0,0.924642,8.37338,0.51458,1



Diverse Counterfactual set (new outcome: 0)


,word_count,sentence_count,avg_sentence_length,sentence_length_std,sentence_length_cv,vocab_richness,uppercase_word_ratio,comma_density,colon_density,semicolon_density,paren_density,flesch,connector_density,label
0,-,-,-,-,-,-,0.236333289,-,-,-,-,30.469072739,-,0.0
1,-,-,-,-,-,-,0.288590434,-,-,-,-,36.894763743,-,0.0
2,-,-,-,-,-,-,0.117247917,-,-,-,-,60.749444063,-,0.0


In [16]:
feat = list(htr.columns)
Xtr = htr.values.astype(float)
Xte = hte.values.astype(float)

# scale so distances aren't dominated by word_count's large range
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)

# ---- candidate counterfactuals = REAL training notes the model confidently calls real ----
real_mask = (np.array(tr_y) == 0)
real_prob = clf_hand.predict_proba(Xtr)[:, 1]          # P(synthetic)
good_real = real_mask & (real_prob < 0.1)              # confidently-real real notes
cand_idx = np.where(good_real)[0]
cand_s = Xtr_s[cand_idx]
print(f"{len(cand_idx)} realistic real-note counterfactual candidates")

# ---- pick the synthetic note to explain (same one DiCE used) ----
syn_rows = np.where((np.array(te_y) == 1) & (pred == 1))[0]
i = int(syn_rows[np.argmax(prob[syn_rows])])
query = Xte[i]
query_s = scaler.transform(query.reshape(1, -1))

# ---- FACE: nearest realistic real note(s) ----
nn = NearestNeighbors(n_neighbors=3).fit(cand_s)
dist, idx = nn.kneighbors(query_s)

print(f"\nOriginal synthetic note  P(synth)={prob[i]:.3f}")
print(f"{'feature':22s}{'original':>12s}{'CF (real note)':>16s}{'change':>12s}")
for rank, j in enumerate(idx[0]):
    cf = Xtr[cand_idx[j]]
    cf_p = clf_hand.predict_proba(cf.reshape(1, -1))[0, 1]
    print(f"\n--- FACE CF{rank}: real note, P(synth)={cf_p:.3f}, scaled dist={dist[0][rank]:.2f} ---")
    for f, o, c in zip(feat, query, cf):
        change = c - o
        mark = "  <==" if abs(change) > 0.5 * (Xtr[:, feat.index(f)].std()) else ""
        print(f"{f:22s}{o:12.3f}{c:16.3f}{change:+12.3f}{mark}")

794 realistic real-note counterfactual candidates

Original synthetic note  P(synth)=0.999
feature                   original  CF (real note)      change

--- FACE CF0: real note, P(synth)=0.003, scaled dist=2.69 ---
word_count                 583.000         759.000    +176.000
sentence_count              46.000          65.000     +19.000
avg_sentence_length         12.891          11.877      -1.014
sentence_length_std         12.403          10.115      -2.288
sentence_length_cv           0.962           0.852      -0.110
vocab_richness               0.724           0.576      -0.148  <==
uppercase_word_ratio         0.033           0.095      +0.062  <==
comma_density                8.784           6.980      -1.804  <==
colon_density                8.322           9.573      +1.251
semicolon_density            0.000           0.798      +0.798  <==
paren_density                0.925           0.997      +0.073
flesch                       8.373          34.254     +25.881  <==
co